In [ ]:
import pandas as pd

roll_no = 170019
digits = [int(x) for x in str(roll_no)]
last_two = digits[-2:]

categories = ["billing", "account", "general"]

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"}
]

personalized_entries = [
    {"question": "how can i update my payment details", "answer": "You can update your payment details from the billing section.", "keywords": "payment update billing details", "category": categories[last_two[0] % 3]},
    {"question": "how do i change my registered mobile number", "answer": "Go to Account Settings and update your registered mobile number.", "keywords": "mobile number account update", "category": categories[last_two[1] % 3]}
]

df = pd.DataFrame(fixed_entries + personalized_entries)
print(df)


In [ ]:
def score_query(query, df):
    words = query.lower().split()
    result = []

    for i, row in df.iterrows():
        text = (row["question"] + " " + row["keywords"]).lower()
        score = sum(word in text for word in words)
        if score > 0:
            result.append([i, row["question"], row["answer"], row["category"], score])

    result = sorted(result, key=lambda x: x[4], reverse=True)

    return pd.DataFrame(result, columns=["index", "question", "answer", "category", "confidence"])

query = input("Enter your query: ")
print(score_query(query, df))


In [ ]:
def same_category(category_name, df):
    return df[df["category"] == category_name][["question", "answer", "keywords", "category"]]

category_name = personalized_entries[0]["category"]
print(same_category(category_name, df))


In [ ]:
index = 0
new_keyword = input("Enter a new keyword: ")

df.loc[index, "keywords"] = df.loc[index, "keywords"] + " " + new_keyword

file_name = str(roll_no) + "_faq_data.csv"
df.to_csv(file_name, index=False)

print(df)
print("Saved as", file_name)


In [ ]:
print(df.groupby("category").size())


In [ ]:
def score_query_tie(query, df):
    words = query.lower().split()
    result = []

    for i, row in df.iterrows():
        text = (row["question"] + " " + row["keywords"]).lower()
        score = sum(word in text for word in words)
        if score > 0:
            result.append([i, row["question"], row["answer"], row["category"], score])

    result = sorted(result, key=lambda x: x[4], reverse=True)
    result = pd.DataFrame(result, columns=["index", "question", "answer", "category", "confidence"])

    if len(result) > 0:
        highest = result["confidence"].max()
        return result[result["confidence"] == highest]

    return result

print("Tie query:")
print(score_query_tie("fee", df))

print("Non-tie query:")
print(score_query_tie("password", df))
